# Macrocyclic Peptide Membrane Permeability Prediction

**Goal:** Predict the PAMPA (Parallel Artificial Membrane Permeability Assay) permeability
of macrocyclic peptides using two classical ML regressors:
- Random Forest Regressor (RFR)
- Support Vector Regressor (SVR)

**Dataset:** CycPeptMPDB — a curated database of experimentally measured membrane
permeability values for cyclic peptides (SMILES + pre-computed RDKit descriptors).

**Splitting strategy:** Murcko Scaffold Split (scaffold-based train/test separation)
to test generalisation to **new molecular scaffolds** not seen during training.
This is the biologically meaningful evaluation — a random split would overestimate
real-world performance because structurally similar molecules would leak between
train and test sets.

**Reference:**
Parameters and featurisation strategy informed by:
> Chen, Z. et al. *CycPeptMPDB: A Comprehensive Database of Membrane Permeability
> of Cyclic Peptides.* J. Chem. Inf. Model. 2023.
> DOI: 10.1021/acs.jcim.3c00054
>
> Maziarka, Ł. et al. *Molecule Attention Transformer.*
> arXiv:2002.08264 (2020). [Molecular attention mechanism reference]


---
## Section 1 — Environment Setup

Install and import all required libraries.
RDKit handles cheminformatics (SMILES parsing, fingerprints, scaffolds).
scikit-learn provides the ML models and metrics.

In [ ]:
# ── Install dependencies (run once in Google Colab / fresh environment) ─────────
# RDKit: the gold-standard cheminformatics toolkit for Python
# scikit-learn is usually pre-installed in Colab
!pip install rdkit --quiet

In [ ]:
# ── Standard library imports ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── RDKit cheminformatics ────────────────────────────────────────────────────────
from rdkit import Chem                        # Core molecule handling
from rdkit.Chem import AllChem                # Morgan fingerprints (ECFP)
from rdkit.Chem.Scaffolds import MurckoScaffold  # Scaffold extraction

# ── scikit-learn: models, preprocessing, evaluation ─────────────────────────────
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("All libraries loaded successfully.")

---
## Section 2 — Data Loading

Load the CycPeptMPDB PAMPA dataset.  
The CSV contains:
- `SMILES` — canonical SMILES string for each cyclic peptide
- `PAMPA` — experimental membrane permeability (the regression target, log-scaled)
- Dozens of pre-computed RDKit 2D physicochemical descriptors
  (MW, LogP, TPSA, H-bond donors/acceptors, rotatable bonds, etc.)

**Colab users:** uncomment the file-upload block below.

In [ ]:
# ── [COLAB ONLY] Interactive file upload ────────────────────────────────────────
# Uncomment these lines if you are running in Google Colab and want to upload
# the CSV manually instead of mounting Google Drive.

# from google.colab import files
# import io
# print("Upload 'CycPeptMPDB_Peptide_Assay_PAMPA.csv' using the button below:")
# uploaded = files.upload()
# filename = list(uploaded.keys())[0]
# df = pd.read_csv(io.BytesIO(uploaded[filename]))
# print(f"Loaded '{filename}' with shape: {df.shape}")

In [ ]:
# ── Standard file load (local / Colab with Drive mounted) ───────────────────────
# If using Google Drive, replace with:
# df = pd.read_csv("/content/drive/MyDrive/CycPeptMPDB_Peptide_Assay_PAMPA.csv")

CSV_PATH = "CycPeptMPDB_Peptide_Assay_PAMPA.csv"
df_raw = pd.read_csv(CSV_PATH)

print(f"Dataset loaded: {df_raw.shape[0]} molecules, {df_raw.shape[1]} columns")
print(f"PAMPA range: {df_raw['PAMPA'].min():.3f} to {df_raw['PAMPA'].max():.3f}")
df_raw[['SMILES', 'PAMPA']].head(3)

---
## Section 3 — Feature Engineering

We use a **two-part feature vector** for each molecule:

### Part A — Pre-computed RDKit 2D Descriptors
These are physicochemical properties already present in the CycPeptMPDB CSV:
molecular weight, LogP, TPSA, H-bond count, ring descriptors, etc.
They encode the **global** chemical properties relevant to passive diffusion.

### Part B — Morgan Fingerprints (ECFP4, 2048 bits)
Extended Connectivity Fingerprints (radius=2, hence ECFP4) encode the
**local structural environment** around every atom.  
- Radius 2 captures atoms up to 4 bonds away from each centre atom.
- 2048 bits is the standard length for complex macrocycles; shorter vectors
  produce excessive bit collisions for large ring systems.
- These were used as primary features in the MAT (Molecular Attention Transformer)
  benchmark comparison (Maziarka et al., 2020).

**Why combine both?** Descriptors capture global drug-likeness properties
while fingerprints capture local substructure — together they provide
complementary information about membrane permeability determinants.

In [ ]:
# ── Work on a clean copy so the raw dataframe stays intact ──────────────────────
df = df_raw.copy()

# ── Extract regression target ────────────────────────────────────────────────────
# PAMPA values are log-transformed effective permeability coefficients
y = df['PAMPA'].values
smiles_list = df['SMILES'].tolist()

# ── Part A: Isolate the pre-computed RDKit descriptor columns ───────────────────
# We exclude all metadata, text fields, and non-PAMPA assay columns.
# Only numeric (float64 / int64) descriptor columns are kept.
EXCLUDE_COLS = [
    # ── Identifiers & provenance ──────────────────────────────────────────────
    'ID', 'Source', 'Year', 'Version',
    'Original_Name_in_Source_Literature',
    'Structurally_Unique_ID',
    'Same_Peptides_ID', 'Same_Peptides_Source',
    'Same_Peptides_Permeability', 'Same_Peptides_Assay',
    # ── Structural string representations ────────────────────────────────────
    'SMILES', 'HELM', 'HELM_URL', 'Sequence',
    # ── Sequence-level descriptors (not atom-level, excluded to avoid overlap) ─
    'Sequence_LogP', 'Sequence_TPSA',
    'Monomer_Length', 'Monomer_Length_in_Main_Chain', 'Molecule_Shape',
    # ── Permeability assay columns (target + other assays) ───────────────────
    'Permeability', 'PAMPA', 'Caco2', 'MDCK', 'RRCK',
    # ── Detection limit flags (binary censoring indicators, not features) ────
    'Detection_Limit_1', 'Detection_Limit_2',
    'R_PAMAP', 'R_Caco2', 'R_MDCK', 'R_RRCK'
]

descriptor_cols = [
    col for col in df.columns
    if col not in EXCLUDE_COLS and df[col].dtype in [np.float64, np.int64]
]

X_descriptors = df[descriptor_cols].copy()
print(f"Part A: {len(descriptor_cols)} pre-computed RDKit descriptors selected.")

# ── Part B: Generate 2048-bit Morgan Fingerprints (ECFP4) ───────────────────────
# Radius = 2  →  ECFP4 (each bit represents a unique circular substructure)
# nBits  = 2048 → reduces collision rate for large macrocyclic ring systems
fps = []
invalid_smiles = 0

for smi in smiles_list:
    mol = Chem.MolFromSmiles(smi)          # Parse SMILES string into RDKit Mol
    if mol is not None:
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=2,      # ECFP4: atoms within 4 bonds contribute to each bit
            nBits=2048     # Bit vector length
        )
        fps.append(np.array(fp))
    else:
        # Fallback: zero vector for any SMILES that RDKit cannot parse
        fps.append(np.zeros(2048))
        invalid_smiles += 1

X_fingerprints = pd.DataFrame(fps, columns=[f'MFP_{i}' for i in range(2048)])
print(f"Part B: Morgan fingerprints generated. Invalid SMILES: {invalid_smiles}")

---
## Section 4 — Feature Concatenation & Data Cleaning

Concatenate Part A (descriptors) and Part B (fingerprints) into a single matrix.

**Cleaning pipeline:**
1. Force all columns to numeric (catches any stray string values)
2. Downcast to `float32` — this deliberately causes `float64` overflow values
   to become `inf`, making them visible and replaceable
3. Replace all `inf`, `-inf`, and `NaN` with `0`

In [ ]:
# ── Concatenate descriptor and fingerprint feature matrices ──────────────────────
X_combined = pd.concat(
    [X_descriptors.reset_index(drop=True),
     X_fingerprints.reset_index(drop=True)],
    axis=1
)

print(f"Combined feature matrix shape: {X_combined.shape}")
print(f"  — {len(descriptor_cols)} descriptor columns")
print(f"  — 2048 Morgan fingerprint bits")

# ── Three-step data cleaning ─────────────────────────────────────────────────────

# Step 1: Coerce everything to numeric (non-parseable → NaN)
X_combined = X_combined.apply(pd.to_numeric, errors='coerce')

# Step 2: Downcast to float32 — values too large for float32 become ±inf,
# which we then catch in step 3. This avoids silent overflow in tree splits.
X_combined = X_combined.astype(np.float32)

# Step 3: Replace ±inf and NaN with 0
X_combined = X_combined.replace([np.inf, -np.inf], np.nan).fillna(0)

# Quick sanity check
assert not X_combined.isnull().any().any(), "NaN values remain after cleaning!"
assert not np.isinf(X_combined.values).any(), "Inf values remain after cleaning!"
print("Data cleaning complete. No NaN or Inf values remain.")

---
## Section 5 — Murcko Scaffold Split (Generalisation to New Scaffolds)

### Why scaffold splitting matters

A **random split** places structurally similar molecules in both train and test sets.
The model then interpolates between known neighbours rather than genuinely
generalising, which inflates reported performance.

A **Murcko scaffold split** groups molecules by their core ring system
(the Bemis-Murcko scaffold). All molecules sharing a scaffold are kept
entirely in either train or test — **never split across both**.  
This tests how well the model predicts permeability for
**completely novel chemical scaffolds** not seen during training,
which is the real challenge in drug discovery.

### Algorithm
1. Extract the Murcko scaffold for every molecule
2. Group molecule indices by scaffold
3. Sort scaffold groups largest→smallest (greedy packing)
4. Assign whole groups to train until 80 % is reached; the rest go to test

In [ ]:
def get_murcko_scaffold(smiles: str) -> str:
    """
    Extract the canonical SMILES of the Bemis-Murcko scaffold for a molecule.
    Returns an empty string if the SMILES is invalid or scaffold extraction fails.

    The Murcko scaffold is the ring system + linker atoms with all
    side chains removed. It represents the core chemical skeleton.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""  # Invalid SMILES → treat as its own singleton group
    try:
        scaffold_mol = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold_mol)  # Canonical form for consistent grouping
    except Exception:
        return ""


# ── Assign a scaffold to each molecule ──────────────────────────────────────────
print("Generating Murcko scaffolds for all molecules...")
df['Scaffold'] = df['SMILES'].apply(get_murcko_scaffold)

unique_scaffolds = df['Scaffold'].nunique()
print(f"Found {unique_scaffolds} unique scaffolds across {len(df)} molecules.")

# ── Group molecule indices by scaffold ──────────────────────────────────────────
scaffold_groups = df.groupby('Scaffold').groups  # {scaffold_smiles: [idx, idx, ...]}

# Sort groups largest → smallest (greedy bin-packing strategy)
scaffolds_sorted = sorted(
    scaffold_groups.keys(),
    key=lambda s: len(scaffold_groups[s]),
    reverse=True
)

# ── Greedy assignment of whole scaffold groups to train / test ───────────────────
train_idx, test_idx = [], []
train_target = int(0.80 * len(df))  # Target: ~80 % train

for scaffold in scaffolds_sorted:
    group_indices = scaffold_groups[scaffold].tolist()
    if len(train_idx) + len(group_indices) <= train_target:
        train_idx.extend(group_indices)  # Entire scaffold group → training set
    else:
        test_idx.extend(group_indices)   # Entire scaffold group → test set

# ── Build train / test matrices ──────────────────────────────────────────────────
X_train_raw = X_combined.iloc[train_idx]
X_test_raw  = X_combined.iloc[test_idx]
y_train      = y[train_idx]
y_test       = y[test_idx]

print(f"\nScaffold split complete:")
print(f"  Train: {len(train_idx):,} molecules ({len(train_idx)/len(df)*100:.1f}%)")
print(f"  Test : {len(test_idx):,} molecules ({len(test_idx)/len(df)*100:.1f}%)")
print(f"  Test scaffolds are 100% unseen during training.")

---
## Section 6 — Model A: Random Forest Regressor (RFR)

Random Forests are an ensemble of decision trees trained on bootstrapped
subsets of the data. Each tree votes independently, and the final prediction
is the average — this reduces variance and overfitting.

**Hyperparameter choices:**
- `n_estimators=100` — 100 trees balances accuracy vs. compute time
- `max_depth=20` — limits tree depth to prevent memorising the training data
- `random_state=42` — ensures reproducibility
- `n_jobs=-1` — uses all available CPU cores

RFR does **not** require feature scaling (trees are invariant to monotone
feature transformations).

In [ ]:
# ── Random Forest Regressor — no scaling required ───────────────────────────────
# Trees split on feature thresholds; the absolute scale of features is irrelevant.
X_train_rf = X_train_raw  # Use the raw (non-scaled) feature matrix
X_test_rf  = X_test_raw

print("Training Random Forest Regressor...")
rf_model = RandomForestRegressor(
    n_estimators=100,    # Number of trees in the ensemble
    max_depth=20,        # Max depth per tree — prevents overfitting
    random_state=42,     # Seed for reproducibility
    n_jobs=-1            # Parallelise across all CPU cores
)
rf_model.fit(X_train_rf, y_train)
print("Training complete.")

# ── Evaluate on the scaffold-held-out test set ───────────────────────────────────
y_pred_rf = rf_model.predict(X_test_rf)

rf_r2  = r2_score(y_test, y_pred_rf)
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_mse = mean_squared_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(rf_mse)

print("\n" + "=" * 45)
print("  Random Forest — Scaffold Test Set Results")
print("=" * 45)
print(f"  R²   : {rf_r2:.4f}")
print(f"  MAE  : {rf_mae:.4f}")
print(f"  RMSE : {rf_rmse:.4f}")
print(f"  MSE  : {rf_mse:.4f}")
print("=" * 45)

---
## Section 7 — Model B: Support Vector Regressor (SVR)

SVR finds a regression function that lies within an ε-tube around the data
while minimising complexity. The RBF (Radial Basis Function) kernel maps
inputs into a higher-dimensional space where non-linear relationships can
be modelled as linear.

**Hyperparameter choices:**
- `kernel='rbf'` — best for high-dimensional, non-linear chemical data
- `C=1.0` — regularisation parameter; higher C = tighter fit to training data
- `epsilon=0.1` — width of the insensitive tube; predictions within ε
  incur no penalty

**Critical:** SVR is a **distance-based** algorithm. Features must be
standardised (zero mean, unit variance) before training. The `StandardScaler`
is fitted **only on the training data** to prevent test-set information
from leaking into the scaling statistics.

In [ ]:
# ── Feature Scaling — mandatory for SVR ─────────────────────────────────────────
# Fit scaler ONLY on training data; transform both train and test.
# This prevents data leakage (test statistics must never influence training).
scaler = StandardScaler()
X_train_svm = scaler.fit_transform(X_train_raw)  # Fit + transform on train
X_test_svm  = scaler.transform(X_test_raw)        # Transform only on test

print(f"Features scaled. Mean ≈ {X_train_svm.mean():.4f}, Std ≈ {X_train_svm.std():.4f}")

# ── Train SVR ────────────────────────────────────────────────────────────────────
print("Training Support Vector Regressor (this may take a few minutes)...")
svm_model = SVR(
    kernel='rbf',    # Radial Basis Function — maps to infinite-dimensional space
    C=1.0,           # Regularisation strength (inverse of margin width)
    epsilon=0.1      # ε-insensitive tube half-width
)
svm_model.fit(X_train_svm, y_train)
print("Training complete.")

# ── Evaluate on the scaffold-held-out test set ───────────────────────────────────
y_pred_svm = svm_model.predict(X_test_svm)

svm_r2  = r2_score(y_test, y_pred_svm)
svm_mae = mean_absolute_error(y_test, y_pred_svm)
svm_mse = mean_squared_error(y_test, y_pred_svm)
svm_rmse = np.sqrt(svm_mse)

print("\n" + "=" * 45)
print("  SVR — Scaffold Test Set Results")
print("=" * 45)
print(f"  R²   : {svm_r2:.4f}")
print(f"  MAE  : {svm_mae:.4f}")
print(f"  RMSE : {svm_rmse:.4f}")
print(f"  MSE  : {svm_mse:.4f}")
print("=" * 45)

---
## Section 8 — Model Comparison Summary

In [ ]:
# ── Side-by-side comparison of both models ──────────────────────────────────────
results = pd.DataFrame({
    'Model':  ['Random Forest', 'SVR (RBF)'],
    'Split':  ['Scaffold', 'Scaffold'],
    'R²':     [round(rf_r2, 4),  round(svm_r2, 4)],
    'MAE':    [round(rf_mae, 4), round(svm_mae, 4)],
    'RMSE':   [round(rf_rmse, 4),round(svm_rmse, 4)],
    'MSE':    [round(rf_mse, 4), round(svm_mse, 4)],
})

print("\n" + "=" * 60)
print("         FINAL MODEL COMPARISON (Scaffold Split)")
print("=" * 60)
print(results.to_string(index=False))
print("=" * 60)
print("\nHigher R² and lower MAE/RMSE indicate better generalisation.")
print("Scaffold split is a conservative, drug-discovery-realistic evaluation.")

---
## Notes on New-Scaffold Generalisation

The Murcko scaffold split used here is the **strictly correct** evaluation
for assessing real-world utility. When performance is lower under scaffold
split than under a random split, it reveals that the model is memorising
structural neighbourhoods rather than learning general permeability rules.

Classical ML models (RFR, SVR) trained on fingerprints are generally
weaker at scaffold generalisation compared to graph neural networks or
attention-based models that can reason about molecular topology directly.
The Molecular Attention Transformer (MAT) approach (Maziarka et al., 2020)
was specifically designed to improve exactly this kind of generalisation.

This notebook provides the classical-ML baseline against which such
architectures are benchmarked.

---
## References

1. Chen, Z. et al. *CycPeptMPDB: A Comprehensive Database of Membrane Permeability
   of Cyclic Peptides.* J. Chem. Inf. Model. **63**, 3026–3036 (2023).  
   DOI: [10.1021/acs.jcim.3c00054](https://doi.org/10.1021/acs.jcim.3c00054)

2. Maziarka, Ł. et al. *Molecule Attention Transformer.* arXiv:2002.08264 (2020).  
   URL: [https://arxiv.org/abs/2002.08264](https://arxiv.org/abs/2002.08264)

3. Bemis, G. W. & Murcko, M. A. *The Properties of Known Drugs. 1. Molecular
   Frameworks.* J. Med. Chem. **39**, 2887–2893 (1996).  
   DOI: [10.1021/jm9602928](https://doi.org/10.1021/jm9602928)

4. Rogers, D. & Hahn, M. *Extended-Connectivity Fingerprints.* J. Chem. Inf. Model.
   **50**, 742–754 (2010). DOI: [10.1021/ci100050t](https://doi.org/10.1021/ci100050t)
